In [ ]:
# libraries imported
import numpy as np
import pandas as pd
import os
import json
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [13]:
# dataset importing

dataset_path = "../backend/data/AQ of adoloscents - Sheet.csv"

df = pd.read_csv(dataset_path)

print(df.head())

   Q1  Q2  Q3  Q4  Q5  Q6  Q7  Q8  Q9  Q10   CONTROL  OWNERSHIP  REACH  \
0   4   5   4   4   5   4   3   4   5    4  4.666667        4.5    3.5   
1   4   4   3   4   4   4   4   4   4    5  4.000000        4.0    3.5   
2   5   5   5   5   3   3   3   2   5    3  4.333333        4.0    4.0   
3   5   5   4   5   5   5   5   5   5    4  5.000000        5.0    4.5   
4   4   4   3   5   4   4   5   4   4    3  4.000000        4.0    4.0   

   ENDURANCE        AQ  Target_Category  
0   4.000000  4.166667                2  
1   4.333333  3.958333                2  
2   3.333333  3.916667                2  
3   4.666667  4.791667                2  
4   4.000000  4.000000                2  


In [14]:
# features and targets

X = df[[f"Q{i}" for i in range(1, 11)]]

y = df["Target_Category"]

print("\nTarget distribution:")
print(y.value_counts())


Target distribution:
Target_Category
2    93
1    10
0     3
Name: count, dtype: int64


In [15]:
# train-test splitting
x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

In [16]:
# feature scaling

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.fit_transform(x_test)

In [17]:
# train SVM model
svm_model = SVC(
    kernel = 'rbf',
    probability = True,
    random_state = 42
)

svm_model.fit(x_train_scaled, y_train)

SVC(probability=True, random_state=42)

In [18]:
y_pred = svm_model.predict(x_test_scaled)

In [19]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test,
    y_pred,
    average='weighted'
)
recall = recall_score(
    y_test,
    y_pred,
    average='weighted'
)
f1 = f1_score(
    y_test,
    y_pred,
    average='weighted'
)

print("\n===== SVM MODEL PERFORMANCE =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


===== SVM MODEL PERFORMANCE =====
Accuracy : 0.8636
Precision: 0.7459
Recall : 0.8636
F1 Score : 0.8004

Classification Report:

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.00      0.00      0.00         2
           2       0.86      1.00      0.93        19

    accuracy                           0.86        22
   macro avg       0.29      0.33      0.31        22
weighted avg       0.75      0.86      0.80        22


Confusion Matrix:

[[ 0  0  1]
 [ 0  0  2]
 [ 0  0 19]]


/mnt/Softs/Miniconda/conda-envs/mini_project/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/mnt/Softs/Miniconda/conda-envs/mini_project/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/mnt/Softs/Miniconda/conda-envs/mini_project/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average,

In [20]:
save_dir = "../backend/ml_models/svm"
os.makedirs(save_dir, exist_ok=True)
# Save model
joblib.dump(
    svm_model,
    os.path.join(save_dir, "svm_model.pkl")
)
# Save scaler
joblib.dump(
    scaler,
    os.path.join(save_dir, "scaler.pkl")
)
# Save metrics
metrics = {
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "model_name": "Support Vector Machine"
}
with open(
    os.path.join(save_dir, "evaluation_metrics.json"),
    "w"
) as f:
    json.dump(metrics, f, indent=4)

print("\nSVM model artifacts saved successfully!")


SVM model artifacts saved successfully!


In [21]:
sample_input = np.array([[
    4, 5, 3, 4, 5,
    4, 3, 4, 5, 4
]])

sample_input_scaled = scaler.transform(sample_input)

prediction = svm_model.predict(sample_input_scaled)[0]

probabilities = svm_model.predict_proba(sample_input_scaled)[0]

category_map = {
    0: "Low",
    1: "Medium",
    2: "High"
}

print("\n===== SAMPLE PREDICTION =====")
print("Predicted Category:",
    category_map[prediction])

print("Confidence Scores:") 

for idx, prob in enumerate(probabilities):
    print(f"{category_map[idx]}: {prob:.4f}")


===== SAMPLE PREDICTION =====
Predicted Category: High
Confidence Scores:
Low: 0.0085
Medium: 0.0175
High: 0.9740


/mnt/Softs/Miniconda/conda-envs/mini_project/lib/python3.11/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
